In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Dijkstra's Shortest Path Algorithm

The file `AVLSet.ts` implements <em style="color:blue">sets</em> as
<a href="https://en.wikipedia.org/wiki/AVL_tree">AVL trees</a>.
The API provided by `AVLSet` offers the following API:
- `AVLSet()` creates an empty set.
- `S.isEmpty()` checks whether the set `S`is empty.
- `S.member(x)` checks whether `x` is an element of the given set `S`.
- `S.insert(x)` inserts `x` into the set `S`.
   This does not return a new set but rather modifies the given set `S`.
- `S.delete(x)` deletes `x` from the set `S`.
   This does not return a new set but rather modifies the set `S`.
- `S.pop()` returns the <em style="color:blue">smallest element</em> of the set `S`.
   Furthermore, this element is removed from the given set `S`.
   
Since sets are implemented as *ordered binary trees*, the elements of a set need to be comparable, i.e. if 
`x` and `y` are inserted into a set, then the expression `x < y` has to be defined and has to return a 
Boolean value.  Furthermore, the relation `<` has to be a 
[linear order](https://en.wikipedia.org/wiki/linear_order).
    
The class `AVLSet` can be used to implement a priority queue that supports the 
*removal* of elements.  Thereby we can change the priority of an element.

In [ ]:
import { AVLSet } from './Set';

The function call `shortestPath` takes a node `source` and a set `Edges`.

The function `shortestPath` takes two arguments.
- `source` is the start node.  
- `Edges` is a dictionary that encodes the set of edges of the graph.  For every node `x` the value of `Edges[x]` has the form
   $$ \bigl[ (y_1, l_1), \cdots, (y_n, l_n) \bigr]. $$
   This list is interpreted as follows: For every $i = 1,\cdots,n$ there is an edge
   $(x, y_i)$ pointing from $x$ to $y_i$ and this edge has the length $l_i$.
   
The function returns the dictionary `Distance`.  For every node `u` such that there is a path from `source` to 
`u`, `Distance[u]` is the length of the shortest path from `source` to `u`.  The implementation uses 
<a href="https://en.wikipedia.org/wiki/Dijkstra%27s_algorithm">Dijkstra's algorithm</a> and proceeds as follows:

- `Distance` is a dictionary mapping nodes to their estimated distance from the node
  `source`.  If `d = Distance[x]`, then we know that there is a path of length `d` leading
  from `source` to `x`.  However, in general we do not know whether there is a path shorter
  than `d` that also connects the source to the node `x`.
- The function `shortestPath` maintains an additional variable called `Visited`.
  This variable contains the set of those nodes that have been  <em style="color:blue">visited</em> 
  by the algorithm.
  To be more precise, `Visited` contains those nodes `u` that have been removed from the
  `Fringe` and for which all neighboring nodes, i.e. those nodes `y` such that
  there is an edge `(u,y)`, have been examined.  It can be shown that once a node `u` is added to
  `Visited`, `Distance[u]` is the length of the shortest path from `source` to `u`.
- `Fringe` is a priority queue that contains pairs of the form `(d, x)`, where `x` is a node and `d`
  is the distance that `x` has from the node `source`.  This priority queue is implemented as a set,
  which in turn is represented by an ordered binary tree.  The fact that we store the node `x` and the
  distance `d` as a pair `(d,x)` implies that the distances are used as priorities because pairs are
  compared lexicographically.
  Initially the only node that is known to be
  reachable from `source` is the node `source`.  Hence `Fringe` is initialized as the
  set `{ (0, source) }`.
- As long as the set `Fringe` is not empty, line 7 of the implementation removes that node `u`
  from the set `Fringe` that has the smallest distance `d` from the node `source`.
- Next, all edges leading away from `u` are visited.  If there is an edge `(u, v)` that has length `l`,
  then we check whether the node `v` has already a distance assigned.  If the node `v` already has the
  distance `dv` assigned but the value `d + l` is less than `dv`, then we have found a
  shorter path from `source` to `v`.  This path leads from `source` to `u` and then proceeds
  to `v` via the edge `(u,v)`.
- If `v` had already been visited before and hence `dv=Distance[v]` is defined, we
  have to update the priority of the `v` in the `Fringe`.  The easiest way to do this is to remove
  the old pair `(dv, v)` from the `Fringe` and replace this pair by the new pair
  `(d+l, v)`, because `d+l` is the new estimate of the distance between `source` and `v` and
  `d+l` is the new priority of `v`.
- Once we have inspected all neighbours of the node `u`, `u` is added to the set of those nodes that have
  been `Visited`.
- When the `Fringe` has been exhausted, the dictionary `Distance` contains the distances of
  every node that is reachable from the node `source`

In [ ]:
function shortestPath(
    source: string,
    Edges: Record<string, Array<[string, number]>>
): Record<string, number> {
    const Distance: Record<string, number> = { [source]: 0 };
    const visited: Record<string, boolean> = {};
    const Fringe = new AVLSet<[number, string]>();
    Fringe.insert([0, source]);
    while (!Fringe.isEmpty()) {
        const [d, u] = Fringe.pop();
        if (visited[u]) {
            continue; 
        }
        visited[u] = true;

        for (const [v, l] of Edges[u]) {
            const dv = Distance[v];
            if (dv === undefined || d + l < dv) {
                if (dv !== undefined) {
                    Fringe.delete([dv, v]);
                }
                Distance[v] = d + l;
                Fringe.insert([d + l, v]);
            }
        }
    }
    return Distance;
}

## Code to Display the Directed Graph

In [ ]:
import { Graphviz } from "@hpcc-js/wasm";
import { display } from "tslab";

The function $\texttt{toDot}(\texttt{source}, \texttt{Edges}, \texttt{Fringe}, \texttt{Distance}, \texttt{Visited})$ takes a graph that is represented by 
its `Edges`, a set of nodes `Fringe`, and a dictionary `Distance` that has the distance of a node from the node `source`,  and set `Visited` of nodes that have already been visited.

In [ ]:
async function toDot(
    source: string,
    p: string | null,
    Edges: Record<string, Array<[string, number]>>,
    Fringe: AVLSet<[number, string]>,
    Distance: Record<string, number>,
    Visited: Record<string, boolean>
): Promise<void> {
    const V = new Set<string>();
    for (const x of Object.keys(Edges)) {
        V.add(x);
    }
    let dot = `digraph G {\nnode [shape=record style=rounded]\nrankdir=LR;\nsize="8,5";\n`;
    for (const x of V) {
        const distanceLabel = Distance[x] !== undefined ? String(Distance[x]) : '';
        if (x === source) {
            dot += `  "${x}" [color=blue shape=doublecircle];\n`;
        } else if (x === p) {
            dot += `  "${x}" [label="{${x}|${distanceLabel}}" color=magenta];\n`;
        } else if (Distance[x] !== undefined && Fringe.member([Distance[x], x])) {
            dot += `  "${x}" [label="{${x}|${distanceLabel}}" color=red];\n`;
        } else if (Visited[x]) {
            dot += `  "${x}" [label="{${x}|${distanceLabel}}" color=blue];\n`;
        } else {
            dot += `  "${x}" [label="{${x}|${distanceLabel}}"];\n`;
        }
    }
    for (const u of V) {
        for (const [v, l] of Edges[u]) {
            dot += `  "${u}" -> "${v}" [label="${l}"];\n`;
        }
    }
    dot += `}`;
    const gv = await Graphviz.load();
    const svg = gv.layout(dot, "svg", "dot");
    const widthMatch = svg.match(/width="([^"]+)"/);
    const heightMatch = svg.match(/height="([^"]+)"/);
    const formattedWidth = widthMatch ? widthMatch[0].replace(/"/g, '').replace('=', ':') : '';
    const formattedHeight = heightMatch ? heightMatch[0].replace(/"/g, '').replace('=', ':') : '';
    const html = `
        <div style='${formattedWidth}; ${formattedHeight};'>
            ${svg}
        </div>
    `;
    display.html(html);
}

The version of `shortestPath` given below provides a graphical animation of the algorithm.

In [ ]:
async function shortestPath(
    source: string,
    Edges: Record<string, Array<[string, number]>>
): Promise<Record<string, number>> {
    const Distance: Record<string, number> = { [source]: 0 };
    const visited: Record<string, boolean> = { [source]: true };
    const Fringe = new AVLSet<[number, string]>();
    Fringe.insert([0, source]);
    while (!Fringe.isEmpty()) {
        const [d, u] = Fringe.pop();
        await toDot(source, u, Edges, Fringe, Distance, visited);
        console.log('_'.repeat(80));
        for (const [v, l] of Edges[u]) {
            const dv = Distance[v];
            if (dv === undefined || d + l < dv) {
                if (dv !== undefined) {
                    Fringe.delete([dv, v]);
                }
                Distance[v] = d + l;
                Fringe.insert([d + l, v]);
            }
        }
        visited[u] = true;
    }
    await toDot(source, null, Edges, Fringe, Distance, visited);
    return Distance;
}

## Code for Testing

In [ ]:
const Edges: Record<string, Array<[string, number]>> = {
    a: [ ['c', 2], ['b', 9] ],
    b: [ ['d', 1] ],
    c: [ ['e', 5], ['g', 3] ],
    d: [ ['f', 2], ['e', 4] ],
    e: [ ['f', 1], ['b', 2] ],
    f: [ ['h', 5] ],
    g: [ ['e', 1] ],
    h: []
};

In [ ]:
const s = 'a';
const sp = await shortestPath(s, Edges);
console.log(sp);

# Crossing the Tunnel

Four people, Alice, Britney, Charly and Daniel have to cross a tunnel.
The tunnel is so narrow, that at most two persons can cross it together.
In order to cross the tunnel, a torch is needed.  Together, they only
have a single torch.  This torch will only last for 12 minutes.
  1. Alice   is the fastest and can cross the tunnel in 1 minute.
  2. Britney needs 2 minutes to cross the tunnel.
  3. Charly  is slower and needs 4 minutes.
  4. Daniel  is the slowest and takes 5 minutes to cross the tunnel.
  
What is the fastest way to cross the tunnel?

We will model this problem as a graph theoretical problem.  The nodes of the graph will be sets 
of people.  In particular, it will be the set of people at the entrance of the tunnel. In order to 
keep track of the torch, the torch can also be a member of these sets. 

In [ ]:
const All: Set<string> = new Set(['Alice', 'Britney', 'Charly', 'Daniel', 'Torch']);

The times needed to cross the tunnel are stored in a dictionary.

In [ ]:
const Time: Record<string, number> = {
  'Alice': 1,
  'Britney': 2,
  'Charly': 4,
  'Daniel': 5,
  'Torch': 0,
};

The function `setToKey` converts a Set of strings into a sorted JSON string. This creates a unique, order-independent key from the set elements.

In [ ]:
function setToKey(s: Set<string>): string {
  return JSON.stringify(Array.from(s).sort());
}

The function $\texttt{power}(M)$ defined below computes the power list of the set $M$, i.e. we have:
$$ \texttt{power}(M) = 2^M = \bigl\{A \mid A \subseteq M \bigr\} $$

In [ ]:
function power(M: Set<string>): Set<Set<string>> {
  if (M.size === 0) {
    return new Set([new Set()]);
  } else {
    const C = new Set(M);
    const iterator = C.values();
    const x = iterator.next().value;
    C.delete(x);
    const P1 = power(C);
    const P2 = new Set<Set<string>>();
    for (const A of P1) {
      const unionSet = new Set(A);
      unionSet.add(x);
      P2.add(unionSet);
    }
    return new Set([...P1, ...P2]);
  }
}

If $B$ is a set of people, then $\texttt{duration}(B)$ is the time that this group needs to cross the tunnel.
$B$ might also contain `'Torch'`.

In [ ]:
function duration(B: Set<string>): number {
  let maxTime = 0;
  B.forEach(x => {
    if (Time[x] !== undefined && Time[x] > maxTime) {
      maxTime = Time[x];
    }
  });
  return maxTime;
}

$\texttt{left\_right}(S)$ describes a crossing of the tunnel from the entrance at the left side left to the exit at the right side of the tunnel.
`S` is the set of people that are initially at the entrance on the left, while `B` is the set op people (together with the torch) that march through
the tunnel.

In [ ]:
function left_right(S: Set<string>): Array<[Set<string>, number]> {
  const result: Array<[Set<string>, number]> = [];
  const pset = power(S);
  pset.forEach(B => {
    if (B.has('Torch') && (B.size >= 2 && B.size <= 3)) {
      const difference = new Set([...S].filter(x => !B.has(x)));
      result.push([difference, duration(B)]);
    }
  });
  return result;
}

$\texttt{right\_left}(S)$ describes a crossing of the tunnel from right to left.

In [ ]:
function right_left(S: Set<string>): Array<[Set<string>, number]> {
  const result: Array<[Set<string>, number]> = [];
  const allMinusS = new Set([...All].filter(x => !S.has(x)));
  const pset = power(allMinusS);
  pset.forEach(B => {
    if (B.has('Torch') && (B.size >= 2 && B.size <= 3)) {
      const unionSet = new Set([...S, ...B]);
      result.push([unionSet, duration(B)]);
    }
  });
  return result;
}

In [ ]:
const Edges: Record<string, Array<[string, number]>> = {};
power(All).forEach(S => {
  const keyS = setToKey(S);
  Edges[keyS] = [];
  for (const [T, w] of left_right(S)) {
    const keyT = setToKey(T);
    Edges[keyS].push([keyT, w]);
  }
  for (const [T, w] of right_left(S)) {
    const keyT = setToKey(T);
    Edges[keyS].push([keyT, w]);
  }
});
console.log(Object.keys(Edges).length);

The function `shortestPath` is Dijkstra's algorithm.  It returns both a dictionary `Parent` containing 
the parent nodes and a dictionary `Distance` with the distances.  The dictionary `Parent` can be used to
compute the shortest path leading from the node `source` to some other node. 

In [ ]:
function shortestPath(
  source: string,
  Edges: Record<string, Array<[string, number]>>
): [Record<string, string | undefined>, Record<string, number>] {
  const Distance: Record<string, number> = { [source]: 0 };
  const Parent: Record<string, string | undefined> = {};
  const Fringe = new AVLSet<[number, string]>();
  Fringe.insert([0, source]);
  while (!Fringe.isEmpty()) {
    const [d, u] = Fringe.pop();
    for (const [v, l] of Edges[u] || []) {
      const dv = Distance[v];
      if (dv === undefined || d + l < dv) {
        if (dv !== undefined) {
          Fringe.delete([dv, v]);
        }
        Distance[v] = d + l;
        Fringe.insert([d + l, v]);
        Parent[v] = u;
      }
    }
  }
  return [Parent, Distance];
}

In [ ]:
const [Parent, Distance] = shortestPath(setToKey(All), Edges);

Let us see whether the goal was reachable and how long it takes to reach the goal.

In [ ]:
const goal = setToKey(new Set<string>());
console.log(Distance[goal]);

Given to nodes `source` and `goal` and a dictionary containing the parent of every node, the function
`findPath` returns the path from `source` to `goal`.

In [ ]:
function findPath(
  source: string,
  goal: string,
  Parent: Record<string, string | undefined>
): string[] {
  const p = Parent[goal];
  if (p === undefined) {
    return [source];
  }
  return [...findPath(source, p, Parent), goal];
}

In [ ]:
const startKey = setToKey(new Set(All));
const goalKey = setToKey(new Set());

const Path = findPath(startKey, goalKey, Parent);

In [ ]:
function printPath(
  Path: string[],
  All: Set<string>,
  duration: (s: Set<string>) => number
): void {
  let total = 0;
  console.log('_'.repeat(81));
  for (let i = 0; i < Path.length; i++) {
    const Left = new Set(JSON.parse(Path[i]) as string[]);
    const Right = new Set([...All].filter(x => !Left.has(x)));
    if (Left.size === 0 || Right.size === 0) {
      console.log(Left, ' '.repeat(25), Right);
    } else {
      console.log(Left, ' '.repeat(30), Right);
    }
    console.log('_'.repeat(81));
    if (i < Path.length - 1) {
      const currentSet = Left;
      const nextSet = new Set(JSON.parse(Path[i + 1]) as string[]);
      if (currentSet.has('Torch')) {
        const Diff = new Set([...currentSet].filter(x => !nextSet.has(x)));
        const time = duration(Diff);
        total += time;
        console.log(' '.repeat(20), '>>>', Diff, ':', time, '>>>');
      } else {
        const Diff = new Set([...nextSet].filter(x => !currentSet.has(x)));
        const time = duration(Diff);
        total += time;
        console.log(' '.repeat(20), '<<<', Diff, ':', time, '<<<');
      }
      console.log('_'.repeat(81));
    }
  }
  console.log('Total time:', total, 'minutes.');
}

In [ ]:
printPath(Path, All, duration);